# <u>Introduction<u/>

We will use the daily data of the past 8 years of the S&P 500 (SPX) for our model, fixing the end date on Jan 20 2026. 

We will set our dependent variable to be labeled as [0,1], with it being the closing price of SPX the next trading day, with the probabilities as follows: 

**Class 1**: $$\text{next day's return} > 0.25\%$$

**Class 0**: $$\text{next day's return} \leq 0.25\%$$

This is to simulate positive up moves only form our model
    
**Methodology** : We will also use the 7 step approach taught in CQF:

| Steps        | Workflow                  | Remarks                                                         |
|:-------------|:--------------------------|:----------------------------------------------------------------|
|Step 1        | Ideation                  | Define objective, success metrics     |
|Step 2        | Data Collection           | Gather and integrate data
|Step 3        | Exploratory Data Analysis (Initial) | Broad exploration: stats, distributions, correlations, missing data |
|Step 4        | Data Cleaning           | Handle missing values, outliers, duplicates.            |
|Step 5        | Feature Engineering & Transformation            | Feature creation, scaling, encoding, selection                         |
|        | Subset Validation EDA            | Re-examine chosen features: check distributions, multicollinearity, relationships                      |               
|Step 6        | Modeling                  | Select algorithm(s), train models, tune hyperparameters                           |
|Step 7        | Evaluation                   | Validate using metrics and backtesting with trading strategy     |
    


Objective: produce a model to predict positive moves (up trend) using the Blending Ensemble
technique.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Data acquisition
import yfinance as yf

# Technical indicators
import ta

# Interactive plotting
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'  # For Jupyter; change to 'browser' if needed

# Preprocessing & Model Selection
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFECV
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Models for ensemble
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve
)

# Variance Inflation Factor
from statsmodels.stats.outliers_influence import variance_inflation_factor

# SHAP for interpretability
import shap

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.style.use('seaborn-v0_8-whitegrid')

print("="*80)
print("SPX ENSEMBLE CLASSIFICATION")
print("="*80)

## STEP 1: IDEATION

In [ ]:
print("\n" + "="*80)
print("STEP 1: IDEATION")
print("="*80)

# Configuration
CONFIG = {
    'ticker': '^GSPC',  # S&P 500
    'end_date': '2026-01-20',
    'lookback_years': 8,
    'threshold': 0.0025,  # 0.25% threshold for Class 1
    'test_size': 0.2,   # Last 20% for final backtest
    'val_size': 0.15,   # 15% for validation
    'random_state': 42,
    'n_splits_cv': 5,   # Time series CV splits

    # Feature selection parameters
    'vif_threshold': None,          # [REDACTED] VIF cutoff
    'correlation_threshold': None,  # [REDACTED] pairwise correlation cutoff
    'mi_top_k': None,               # [REDACTED] top-K by mutual information
    'n_clusters': None,             # [REDACTED] K-Means cluster count
    'rfecv_min_features': None      # [REDACTED] RFECV minimum features
}

print(f"""
Objective: Binary classification to predict SPX up moves > {CONFIG['threshold']*100}%

Target Variable:
  - Class 1: Next day return > {CONFIG['threshold']*100}%
  - Class 0: Next day return <= {CONFIG['threshold']*100}%

Success Metrics:
  - Classification: Precision, Recall, F1, AUC-ROC
  - Trading: Sharpe Ratio, Max Drawdown, Win Rate

Ensemble Architecture:
  - Base Learners: XGBoost, LightGBM, Random Forest, Logistic Regression (Elastic Net)
  - Meta-Model: Logistic Regression (Ridge)
  - Validation: Walk-forward time series cross-validation

Feature Selection Pipeline (Funnel Approach):
  - Stage 1 (Filter): VIF, correlation, Mutual Information top-K
  - Stage 2 (Cluster): K-Means, select representative per cluster
  - Stage 3 (Wrapper): RFECV with LightGBM
  - Interpretability: SHAP analysis
""")


## STEP 2: DATA COLLECTION

In [ ]:
print("\n" + "="*80)
print("STEP 2: DATA COLLECTION")
print("="*80)

def collect_data(ticker, end_date, years):
    """Download OHLCV data from Yahoo Finance."""
    end = pd.to_datetime(end_date)
    start = end - timedelta(days=years*365 + 100)  # Extra days for warmup
    
    print(f"Downloading {ticker} data from {start.date()} to {end.date()}...")
    
    df = yf.download(ticker, start=start, end=end, progress=False)
    
    # Flatten multi-level columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    print(f"Downloaded {len(df)} trading days")
    print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
    
    return df

# Download SPX data
df_raw = collect_data(CONFIG['ticker'], CONFIG['end_date'], CONFIG['lookback_years'])

# Also download VIX for volatility regime
print("\nDownloading VIX for volatility regime indicator...")
vix = yf.download('^VIX', start=df_raw.index[0], end=df_raw.index[-1], progress=False) #download progress bar doesnt show
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)
df_raw['VIX'] = vix['Close'] #adds column of VIX close data

print(f"\nRaw data shape: {df_raw.shape}")
print(df_raw.head())

## STEP 3: EXPLORATORY DATA ANALYSIS (INITIAL)

In [ ]:
# ==============================================================================
# STEP 3: EXPLORATORY DATA ANALYSIS (INITIAL)
# ==============================================================================
print("\n" + "="*80)
print("STEP 3: EXPLORATORY DATA ANALYSIS (INITIAL)")
print("="*80)

def initial_eda(df):
    """Perform initial exploratory data analysis."""
    print("\n--- Basic Statistics ---")
    print(df.describe())
    
    print("\n--- Missing Values ---")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.sum() > 0 else "No missing values in OHLCV") #missing data col printed
    
    # Calculate returns
    df['Returns'] = df['Close'].pct_change() #add new return col
    
    print("\n--- Returns Distribution ---")
    print(f"Mean daily return: {df['Returns'].mean()*100:.4f}%")
    print(f"Std daily return: {df['Returns'].std()*100:.4f}%")
    print(f"Skewness: {df['Returns'].skew():.4f}")
    print(f"Kurtosis: {df['Returns'].kurtosis():.4f}")
    
    # Target distribution preview
    threshold = CONFIG['threshold']
    next_day_return = df['Returns'].shift(-1) #shift series up by 1 row
    class_1_pct = (next_day_return > threshold).mean() * 100 #if tmr return >0.25% then 1 - .mean() iss a boolean *100 converts to %
    print(f"\n--- Target Class Balance Preview ---")
    print(f"Class 1 (return > {threshold*100}%): {class_1_pct:.1f}%")
    print(f"Class 0 (return <= {threshold*100}%): {100-class_1_pct:.1f}%")
    
    return df

df_raw = initial_eda(df_raw)

# Interactive Visualizations using Plotly
print("\n--- Interactive EDA Plots (Plotly) ---")

# Create subplot figure
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('SPX Price History', 'Daily Returns Distribution', 
                    'VIX (Volatility Index)', '21-Day Rolling Annualized Volatility (%)'),
    specs=[[{"type": "scatter"}, {"type": "histogram"}],
           [{"type": "scatter"}, {"type": "scatter"}]]
)

# Price history
fig.add_trace(
    go.Scatter(x=df_raw.index, y=df_raw['Close'], mode='lines', name='SPX Close', #line chart
               line=dict(color='blue')),
    row=1, col=1
)

# Returns distribution
returns_clean = df_raw['Returns'].dropna()
fig.add_trace(
    go.Histogram(x=returns_clean, nbinsx=100, name='Returns', 
                 marker_color='steelblue', opacity=0.7),
    row=1, col=2
)
fig.add_vline(x=CONFIG['threshold'], line_dash="dash", line_color="red", #line
              annotation_text=f"Threshold ({CONFIG['threshold']*100}%)", row=1, col=2)
fig.add_vline(x=0, line_dash="dash", line_color="green", row=1, col=2) #line

# VIX history
fig.add_trace(
    go.Scatter(x=df_raw.index, y=df_raw['VIX'], mode='lines', name='VIX',
               line=dict(color='orange')),
    row=2, col=1
)

# Rolling volatility
rolling_vol = df_raw['Returns'].rolling(21).std() * np.sqrt(252) * 100
fig.add_trace(
    go.Scatter(x=df_raw.index, y=rolling_vol, mode='lines', name='Rolling Vol',
               line=dict(color='purple')),
    row=2, col=2
)

fig.update_layout(
    height=800, width=1200,
    title_text="Step 3: Initial EDA",
    showlegend=False
)
fig.show()

## STEP 4: DATA CLEANING

In [ ]:
# ==============================================================================
# STEP 4: DATA CLEANING
# ==============================================================================
print("\n" + "="*80)
print("STEP 4: DATA CLEANING")
print("="*80)

def clean_data(df):
    """Handle missing values and outliers."""
    df_clean = df.copy()
    
    print(f"Initial shape: {df_clean.shape}") #(rows,cols)
    
    # Handle missing VIX values (forward fill)
    vix_missing = df_clean['VIX'].isnull().sum()
    if vix_missing > 0:
        print(f"VIX missing values: {vix_missing} (forward filling)")
        df_clean['VIX'] = df_clean['VIX'].ffill() #forward fill
    
    # Drop any remaining rows with missing OHLCV
    initial_len = len(df_clean) #store current rows
    df_clean = df_clean.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume']) #drop any missing row
    dropped = initial_len - len(df_clean) #see how many rows dropped
    if dropped > 0:
        print(f"Dropped {dropped} rows with missing OHLCV data")
    
    # Identify extreme outliers in returns (> 5 std)
    returns = df_clean['Returns'].dropna() #removes any row that are empty
    mean_ret = returns.mean() #average daily return
    std_ret = returns.std()
    outliers = np.abs(returns - mean_ret) > 5 * std_ret 
    n_outliers = outliers.sum()
    print(f"\nExtreme outliers (>5 std): {n_outliers} days")
    if n_outliers > 0:
        print("Outlier dates:")
        print(returns[outliers])
    
    # We keep outliers but flag them (market events are real) -> returns not NaN and row is an outlier
    df_clean['Outlier_Flag'] = 0 #new col
    df_clean.loc[df_clean['Returns'].notna() & #select rows and column and had valid return
                 outliers.reindex(df_clean.index, fill_value=False), #aligns outliers mask to full data frame; missing dates get false
                 'Outlier_Flag'] = 1
    
    print(f"\nFinal shape: {df_clean.shape}")
    
    return df_clean

df_clean = clean_data(df_raw)

## STEP 5: FEATURE ENGINEERING & TRANSFORMATION

In [ ]:
# ==============================================================================
# STEP 5: FEATURE ENGINEERING & TRANSFORMATION
# ==============================================================================
print("\n" + "="*80)
print("STEP 5: FEATURE ENGINEERING & TRANSFORMATION")
print("="*80)

# ------------------------------------------------------------------------------
# 150+ features engineered across 7 categories:
#
#   1. Momentum / Trend       — ...
#   2. Volatility Estimators  — ...
#   3. Volume-based           — ...
#   4. Market Microstructure  — ...
#   5. Regime-aware           — ...
#   6. Lag Features           — ...
#   7. Rolling Statistics     — ...
#
# Implementation intentionally omitted.
# ------------------------------------------------------------------------------

# df_features = engineer_features(df_clean)  # [REDACTED]

# ------------------------------------------------------------------------------
# Post-engineering steps (apply after engineer_features):
#   - Drop 252-bar warmup period (longest lookback feature)
#   - Drop rows with NaN target (last row has no next-day return)
#   - Exclude raw/base columns (raw MAs, raw ATR, raw OBV etc.) — keep derived
#     ratio/percentage forms only to avoid scale drift and duplicate information
#   - Forward-fill then backward-fill any remaining NaNs
#   - Replace inf/-inf created by ratio features with NaN, then fill
# ------------------------------------------------------------------------------

### TRAIN/VAL/TEST SPLIT

In [ ]:
# ==============================================================================
# TRAIN/VAL/TEST SPLIT (Before Feature Selection to Prevent Leakage)
# ==============================================================================
print("\n" + "-"*40)
print("TRAIN/VAL/TEST SPLIT")
print("-"*40)

# Prepare data in numpy array
X_all = df_features[feature_cols].values
y_all = df_features['Target'].values
dates_all = df_features.index

# Train/Validation/Test split (time-based)
n_samples = len(X_all)
test_start_idx = int(n_samples * (1 - CONFIG['test_size'])) #last 20% -> 1-20% start at 80
val_start_idx = int(n_samples * (1 - CONFIG['test_size'] - CONFIG['val_size'])) #1-0.2-0.15 =0.65 start

X_train_raw = X_all[:val_start_idx] # start to val start but not inlc val start
y_train = y_all[:val_start_idx]
dates_train = dates_all[:val_start_idx]

X_val_raw = X_all[val_start_idx:test_start_idx]
y_val = y_all[val_start_idx:test_start_idx]
dates_val = dates_all[val_start_idx:test_start_idx]

X_test_raw = X_all[test_start_idx:]
y_test = y_all[test_start_idx:]
dates_test = dates_all[test_start_idx:]

print(f"\nData splits (before scaling):")
print(f"  Training: {len(X_train_raw)} samples ({dates_train[0].date()} to {dates_train[-1].date()})")
print(f"  Validation: {len(X_val_raw)} samples ({dates_val[0].date()} to {dates_val[-1].date()})")
print(f"  Test: {len(X_test_raw)} samples ({dates_test[0].date()} to {dates_test[-1].date()})")


# ==============================================================================
# SCALING (Multiple Scalers for Different Purposes) - split before scaling prevents leakage
# ==============================================================================
print("\n" + "-"*40)
print("SCALING (Context-Appropriate)")
print("-"*40)

print("""
Scaling Strategy:
  - RobustScaler: For EDA, VIF, Correlation (handles outliers in financial data)
  - StandardScaler: For K-Means clustering (distance-based, assumes Gaussian)
  - StandardScaler: For Logistic Regression (gradient-based optimization)
  - No scaling: For tree-based models (XGBoost, LightGBM, RF - scale invariant)
""")

# 1. RobustScaler - for EDA and filter-based feature selection (handles outliers)
robust_scaler = RobustScaler()
X_train_robust = robust_scaler.fit_transform(X_train_raw)
X_val_robust = robust_scaler.transform(X_val_raw)
X_test_robust = robust_scaler.transform(X_test_raw)

# 2. StandardScaler - for K-Means clustering and Logistic Regression
standard_scaler = StandardScaler()
X_train_standard = standard_scaler.fit_transform(X_train_raw)
X_val_standard = standard_scaler.transform(X_val_raw)
X_test_standard = standard_scaler.transform(X_test_raw)

# 3. Unscaled data - for tree-based models (they're scale-invariant)
X_train_unscaled = X_train_raw.copy()
X_val_unscaled = X_val_raw.copy()
X_test_unscaled = X_test_raw.copy()

# Create scaled DataFrames for EDA (using RobustScaler)
df_train_scaled = pd.DataFrame(X_train_robust, columns=feature_cols, index=dates_train)
df_train_scaled['Target'] = y_train

print(f"Scaling complete:")
print(f"  RobustScaler fitted on training data - for EDA/filtering")
print(f"  StandardScaler fitted on training data - for K-Means/Logistic Regression")
print(f"  Unscaled data preserved - for tree-based models")


# ==============================================================================
# SUBSET VALIDATION EDA (On Scaled Features)
# ==============================================================================
print("\n" + "-"*40)
print("SUBSET VALIDATION EDA (SCALED FEATURES)")
print("-"*40)

# Distribution of scaled features (using RobustScaler for EDA) - Interactive Plotly
print("\n--- Scaled Feature Distributions (Interactive) ---")

sample_features = ['RSI_14', 'Vol_CC_21', 'MACD', 'VIX_Ratio_20']
sample_features = [f for f in sample_features if f in feature_cols][:4] #incase missing values, dropped from list with max 4

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{feat} (RobustScaler)' for feat in sample_features]
)

colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

for i, feat in enumerate(sample_features): #gives (i,feat) like (0,RSI_14)
    feat_idx = feature_cols.index(feat) #finds integer position on the feature list
    row, col = (i // 2) + 1, (i % 2) + 1 #simple equation to show where it goes on subplot, i =0-3 -> integer division, modulo ("numerator remainder after division")
    fig.add_trace(
        go.Histogram(x=X_train_robust[:, feat_idx], nbinsx=50, name=feat,
                     marker_color=colors[i], opacity=0.7),
        row=row, col=col
    )

fig.update_layout(
    height=600, width=1000,
    title_text="Step 5: Scaled Feature Distributions (RobustScaler)",
    showlegend=False
)
fig.show()

### FUNNEL FEATURE SELECTION

#### Stage 1

In [ ]:
# ==============================================================================
# FUNNEL FEATURE SELECTION
# ==============================================================================
print("\n" + "="*80)
print("FUNNEL FEATURE SELECTION")
print("="*80)

# ============================================================================
# STAGE 1: FILTER (VIF + Correlation + Mutual Information)
# ============================================================================
print("\n--- STAGE 1: FILTER ---")

# 1a. VIF Analysis
print("\n[1a] Variance Inflation Factor Analysis (RobustScaler)...")

def calculate_vif(X, feature_names, threshold):
    """
    Compute VIF for each feature and return a sorted DataFrame.
    Features with VIF above threshold are flagged as multicollinear.
    """
    # [REDACTED — VIF loop implementation omitted]
    raise NotImplementedError("Implement VIF calculation using statsmodels.stats.outliers_influence.variance_inflation_factor")

vif_df = calculate_vif(X_train_robust, feature_cols, CONFIG['vif_threshold'])
high_vif = vif_df[vif_df['VIF'] > CONFIG['vif_threshold']]
print(f"Features with VIF > threshold: {len(high_vif)}")

vif_pass_features = vif_df[vif_df['VIF'] <= CONFIG['vif_threshold']]['Feature'].tolist()
print(f"Features passing VIF filter: {len(vif_pass_features)}")

# 1b. Correlation Analysis
print("\n[1b] Correlation Analysis (RobustScaler)...")

# [REDACTED — pairwise correlation removal logic omitted]
# Strategy: for each highly correlated pair, drop the feature with higher average
# absolute correlation to all other features.
corr_pass_features = []  # [REDACTED — populate from your correlation filter]

print(f"Features passing correlation filter: {len(corr_pass_features)}")

# 1c. Mutual Information
print("\n[1c] Mutual Information Analysis...")

corr_pass_idx = [feature_cols.index(f) for f in corr_pass_features]
X_train_corr = X_train_robust[:, corr_pass_idx]

mi_scores = mutual_info_classif(X_train_corr, y_train, random_state=CONFIG['random_state'])
mi_df = pd.DataFrame({
    'Feature': corr_pass_features,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

print(f"\nTop features by Mutual Information:")
print(mi_df.head(10))

# Keep top-K by MI
stage1_features = mi_df.head(CONFIG['mi_top_k'])['Feature'].tolist()
print(f"\nFeatures after Stage 1 (Filter): {len(stage1_features)}")


#### Stage 2

In [ ]:
# ============================================================================
# STAGE 2: CLUSTER (K-Means on Features)
# Using StandardScaler - K-Means is distance-based
# ============================================================================
print("\n--- STAGE 2: CLUSTER (K-Means with StandardScaler) ---")

# Get StandardScaler data for stage 1 features
stage1_idx = [feature_cols.index(f) for f in stage1_features]
X_train_stage1_standard = X_train_standard[:, stage1_idx]

# K-Means clustering on FEATURES (transpose so features are samples)
n_clusters = min(CONFIG['n_clusters'], max(2, len(stage1_features) // 2))
kmeans = KMeans(n_clusters=n_clusters, random_state=CONFIG['random_state'], n_init=10)
cluster_labels = kmeans.fit_predict(X_train_stage1_standard.T)

# Feature -> Cluster mapping
cluster_df = pd.DataFrame({
    'Feature': stage1_features,
    'Cluster': cluster_labels
})

# Pick representative per cluster = highest MI feature inside that cluster
cluster_representatives = []
for cluster_id in range(n_clusters):
    cluster_features = cluster_df.loc[cluster_df['Cluster'] == cluster_id, 'Feature'].tolist()

    cluster_mi = (
        mi_df[mi_df['Feature'].isin(cluster_features)]
        .sort_values('MI_Score', ascending=False)
    )

    if len(cluster_mi) > 0:
        best_feature = cluster_mi.iloc[0]['Feature']
        cluster_representatives.append({
            'Cluster': cluster_id,
            'Representative': best_feature,
            'MI_Score': float(cluster_mi.iloc[0]['MI_Score']),
            'Cluster_Size': len(cluster_features),
            'Cluster_Features': cluster_features
        })

cluster_rep_df = pd.DataFrame(cluster_representatives).sort_values('Cluster')
print(f"\nCluster Representatives (k={n_clusters}):")
print(cluster_rep_df[['Cluster', 'Representative', 'MI_Score', 'Cluster_Size']])

stage2_features = cluster_rep_df['Representative'].tolist()
print(f"\nFeatures after clustering: {len(stage2_features)}")

# ============================
# CLEAN CLUSTER VISUALIZATION
# ============================

# Build plot dataframe (THIS was missing in your version)
cluster_size_map = cluster_rep_df.set_index('Cluster')['Cluster_Size'].to_dict()

plot_rows = []
for _, row in cluster_rep_df.iterrows():
    c_id = row['Cluster']
    rep = row['Representative']
    for feat in row['Cluster_Features']:
        mi_val = float(mi_df.loc[mi_df['Feature'] == feat, 'MI_Score'].values[0])
        plot_rows.append({
            'Cluster': int(c_id),
            'Feature': feat,
            'MI_Score': mi_val,
            'Is_Representative': feat == rep,
            'Cluster_Size': int(cluster_size_map[int(c_id)]),
            'Size': 22 if feat == rep else 10
        })

cluster_plot_df = pd.DataFrame(plot_rows)

# Base scatter: color = cluster size (meaningful)
fig = px.scatter(
    cluster_plot_df,
    x='Cluster',
    y='MI_Score',
    color='Cluster_Size',
    size='Size',
    hover_data=['Feature', 'Cluster_Size'],
    title='Feature Clusters (★ = Representative)'
)

# Add reps as star markers with outline
rep_mask = cluster_plot_df['Is_Representative'] == True
fig.add_scatter(
    x=cluster_plot_df.loc[rep_mask, 'Cluster'],
    y=cluster_plot_df.loc[rep_mask, 'MI_Score'],
    mode='markers',
    marker=dict(size=20, symbol='star', line=dict(width=2, color='black')),
    name='Representative',
    text=cluster_plot_df.loc[rep_mask, 'Feature'],
    hoverinfo='text+y'
)

# Layout polish
fig.update_layout(
    height=600,
    width=950,
    coloraxis_colorbar=dict(title='Cluster Size'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(l=60, r=40, t=80, b=60)
)

fig.update_xaxes(dtick=1, title='Cluster ID')
fig.update_yaxes(title='Mutual Information (MI)')

fig.show()


Each cluster groups highly similar / redundant features

From each cluster, one representative feature was selected

Representatives were chosen solely by highest Mutual Information (MI) within their cluster. Vertical stacking indicates redudancy. 

#### Stage 3

In [ ]:
# ============================================================================
# STAGE 3: WRAPPER (RFECV with LightGBM)
# Using unscaled data - LightGBM is tree-based and scale-invariant
# ============================================================================
print("\n--- STAGE 3: WRAPPER (RFECV with Unscaled Data) ---")
print("Note: Using unscaled data - LightGBM is tree-based and scale-invariant")

# Get unscaled data for stage 2 features (tree-based models are scale-invariant)
stage2_idx = [feature_cols.index(f) for f in stage2_features]
X_train_stage2_unscaled = X_train_unscaled[:, stage2_idx]

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=CONFIG['n_splits_cv'])

# LightGBM as base estimator for RFECV (fast and handles imbalance)
lgb_rfe = LGBMClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=CONFIG['random_state'],
    verbose=-1
)

# RFECV
print("Running RFECV (this may take a few minutes)...")
rfecv = RFECV(
    estimator=lgb_rfe,
    step=1,
    cv=tscv,
    scoring='roc_auc',
    # min_features_to_select=CONFIG['rfecv_min_features'],
    n_jobs=-1
)
rfecv.fit(X_train_stage2_unscaled, y_train)

# Get selected features
rfecv_mask = rfecv.support_ #store as boolean mask
stage3_features = [f for f, selected in zip(stage2_features, rfecv_mask) if selected] #pair position by position

print(f"\nRFECV Results:")
print(f"  Optimal number of features: {rfecv.n_features_}")
print(f"  Features selected: {stage3_features}")

# Plot RFECV scores - Interactive Plotly
print("\n--- RFECV Performance (Interactive) ---")

n_features_range = list(range(CONFIG['rfecv_min_features'], len(stage2_features) + 1))
mean_scores = rfecv.cv_results_['mean_test_score']
std_scores = rfecv.cv_results_['std_test_score']

fig = go.Figure()

# Add mean score line
fig.add_trace(go.Scatter(
    x=n_features_range, y=mean_scores,
    mode='lines+markers',
    name='Mean CV Score',
    line=dict(color='blue')
))

# Add confidence band
fig.add_trace(go.Scatter(
    x=n_features_range + n_features_range[::-1],
    y=list(mean_scores + std_scores) + list(mean_scores - std_scores)[::-1],
    fill='toself',
    fillcolor='rgba(0,100,255,0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±1 Std Dev'
))

# Add optimal line
fig.add_vline(x=rfecv.n_features_, line_dash="dash", line_color="red",
              annotation_text=f"Optimal: {rfecv.n_features_}")

fig.update_layout(
    title='RFECV: Feature Selection Performance',
    xaxis_title='Number of Features',
    yaxis_title='CV ROC-AUC Score',
    height=500, width=800
)
fig.show()

# Final selected features
selected_features = stage3_features
print(f"\n--- FUNNEL SELECTION COMPLETE ---")
print(f"  Initial features: {len(feature_cols)}")
print(f"  After Stage 1 (Filter): {len(stage1_features)}")
print(f"  After Stage 2 (Cluster): {len(stage2_features)}")
print(f"  After Stage 3 (Wrapper): {len(selected_features)}")
print(f"\nFinal selected features: {selected_features}")


In [ ]:
# ==============================================================================
# PREPARE FINAL DATA (Different Scalings for Different Models)
# ==============================================================================
print("\n" + "-"*40)
print("PREPARING FINAL DATA (Model-Specific Scaling)")
print("-"*40)

# Get final feature indices
final_idx = [feature_cols.index(f) for f in selected_features]

# Unscaled data for tree-based models (XGBoost, LightGBM, Random Forest)
X_train_trees = X_train_unscaled[:, final_idx]
X_val_trees = X_val_unscaled[:, final_idx]
X_test_trees = X_test_unscaled[:, final_idx]

# StandardScaler data for Logistic Regression (gradient-based optimization)
X_train_lr = X_train_standard[:, final_idx]
X_val_lr = X_val_standard[:, final_idx]
X_test_lr = X_test_standard[:, final_idx]

# RobustScaler data for general analysis/visualization
X_train_robust_final = X_train_robust[:, final_idx]
X_val_robust_final = X_val_robust[:, final_idx]
X_test_robust_final = X_test_robust[:, final_idx]

print(f"""
Final data prepared with model-specific scaling:
  - Tree models (XGBoost, LightGBM, RF): Unscaled data
    Shape: {X_train_trees.shape}
  - Logistic Regression: StandardScaler
    Shape: {X_train_lr.shape}
  - Visualization/Analysis: RobustScaler
    Shape: {X_train_robust_final.shape}
""")

# Correlation heatmap for final features (using RobustScaler for visualization) - Interactive
print("\n--- Final Feature Correlation (Interactive) ---")

corr_final = np.corrcoef(X_train_robust_final.T) #sanity check
corr_final_df = pd.DataFrame(corr_final, index=selected_features, columns=selected_features)

fig = px.imshow(
    corr_final_df,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Final Feature Correlation Matrix'
)
fig.update_layout(height=700, width=800)
fig.show()

In [ ]:
# ==============================================================================
# STEP 6: MODELING
# ==============================================================================
print("\n" + "="*80)
print("STEP 6: MODELING")
print("="*80)

print("\n--- Training Base Learners (Model-Specific Scaling) ---")

def train_base_learners(X_train_trees, X_train_lr, y_train,
                        X_val_trees, X_val_lr, y_val, tscv):
    """
    Train and tune base learners for the ensemble.

    Scaling strategy:
    - Tree-based models (XGBoost, LightGBM, RF): Unscaled data
    - Logistic Regression: StandardScaler data
    """
    base_learners = {}
    val_predictions = {}

    # -------------------------------------------------------------------------
    # 1. XGBoost (Unscaled — tree-based, scale-invariant)
    # -------------------------------------------------------------------------
    print("\n1. XGBoost (Unscaled data — tree-based, scale-invariant)...")

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    xgb_params = {
        'max_depth': [],          # [REDACTED]
        'learning_rate': [],      # [REDACTED]
        'n_estimators': [],       # [REDACTED]
        'min_child_weight': []    # [REDACTED]
    }

    xgb = XGBClassifier(
        objective='binary:logistic',
        scale_pos_weight=scale_pos_weight,
        random_state=CONFIG['random_state'],
        use_label_encoder=False,
        eval_metric='logloss',
        verbosity=0
    )

    xgb_grid = GridSearchCV(xgb, xgb_params, cv=tscv, scoring='roc_auc', n_jobs=-1)
    xgb_grid.fit(X_train_trees, y_train)

    base_learners['xgboost'] = xgb_grid.best_estimator_
    val_predictions['xgboost'] = xgb_grid.best_estimator_.predict_proba(X_val_trees)[:, 1]
    print(f"   Best params: {xgb_grid.best_params_}")
    print(f"   CV AUC: {xgb_grid.best_score_:.4f}")

    # -------------------------------------------------------------------------
    # 2. LightGBM (Unscaled — tree-based, scale-invariant)
    # -------------------------------------------------------------------------
    print("\n2. LightGBM (Unscaled data — tree-based, scale-invariant)...")

    lgb_params = {
        'max_depth': [],          # [REDACTED]
        'learning_rate': [],      # [REDACTED]
        'n_estimators': [],       # [REDACTED]
        'num_leaves': []          # [REDACTED]
    }

    lgb = LGBMClassifier(
        objective='binary',
        class_weight='balanced',
        random_state=CONFIG['random_state'],
        verbose=-1
    )

    lgb_grid = GridSearchCV(lgb, lgb_params, cv=tscv, scoring='roc_auc', n_jobs=-1)
    lgb_grid.fit(X_train_trees, y_train)

    base_learners['lightgbm'] = lgb_grid.best_estimator_
    val_predictions['lightgbm'] = lgb_grid.best_estimator_.predict_proba(X_val_trees)[:, 1]
    print(f"   Best params: {lgb_grid.best_params_}")
    print(f"   CV AUC: {lgb_grid.best_score_:.4f}")

    # -------------------------------------------------------------------------
    # 3. Random Forest (Unscaled — tree-based, scale-invariant)
    # -------------------------------------------------------------------------
    print("\n3. Random Forest (Unscaled data — tree-based, scale-invariant)...")

    rf_params = {
        'n_estimators': [],       # [REDACTED]
        'max_depth': [],          # [REDACTED]
        'min_samples_leaf': [],   # [REDACTED]
        'max_features': []        # [REDACTED]
    }

    rf = RandomForestClassifier(
        class_weight='balanced',
        random_state=CONFIG['random_state']
    )

    rf_grid = GridSearchCV(rf, rf_params, cv=tscv, scoring='roc_auc', n_jobs=-1)
    rf_grid.fit(X_train_trees, y_train)

    base_learners['random_forest'] = rf_grid.best_estimator_
    val_predictions['random_forest'] = rf_grid.best_estimator_.predict_proba(X_val_trees)[:, 1]
    print(f"   Best params: {rf_grid.best_params_}")
    print(f"   CV AUC: {rf_grid.best_score_:.4f}")

    # -------------------------------------------------------------------------
    # 4. Logistic Regression (StandardScaler — gradient-based optimization)
    # -------------------------------------------------------------------------
    print("\n4. Logistic Regression (StandardScaler — gradient-based optimization)...")

    lr_params = {
        'C': [],          # [REDACTED]
        'l1_ratio': []    # [REDACTED]
    }

    lr = LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        class_weight='balanced',
        max_iter=1000,
        random_state=CONFIG['random_state']
    )

    lr_grid = GridSearchCV(lr, lr_params, cv=tscv, scoring='roc_auc', n_jobs=-1)
    lr_grid.fit(X_train_lr, y_train)

    base_learners['logistic'] = lr_grid.best_estimator_
    val_predictions['logistic'] = lr_grid.best_estimator_.predict_proba(X_val_lr)[:, 1]
    print(f"   Best params: {lr_grid.best_params_}")
    print(f"   CV AUC: {lr_grid.best_score_:.4f}")

    return base_learners, val_predictions

base_learners, val_predictions = train_base_learners(
    X_train_trees, X_train_lr, y_train,
    X_val_trees, X_val_lr, y_val, tscv
)


In [ ]:
# ==============================================================================
# META-MODEL (BLENDING)
# ==============================================================================
print("\n--- Training Meta-Model (Blending) ---")

# Stack base-learner validation probabilities into meta-features
meta_X_val = np.column_stack([val_predictions[name] for name in base_learners.keys()])

# Ridge Logistic Regression as meta-learner (trained on validation set only)
meta_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=CONFIG['random_state']
)
meta_model.fit(meta_X_val, y_val)

print(f"\nMeta-model coefficients (base learner weights):")
for name, coef in zip(base_learners.keys(), meta_model.coef_[0]):
    print(f"  {name}: {coef:.4f}")

# ==============================================================================
# ENSEMBLE PREDICTIONS
# ==============================================================================
print("\n--- Generating Ensemble Predictions ---")

def ensemble_predict(base_learners, meta_model, X_trees, X_lr):
    """
    Blending ensemble: collect base-learner probabilities, pass through meta-model.
    Tree models use unscaled data; Logistic Regression uses StandardScaler data.
    Returns ensemble probabilities and individual base-learner predictions.
    """
    # [REDACTED — base-learner call order and stacking logic omitted]
    raise NotImplementedError("Implement base-learner predict_proba calls and meta-model stacking")

val_proba,  val_base_preds  = ensemble_predict(base_learners, meta_model, X_val_trees,  X_val_lr)
test_proba, test_base_preds = ensemble_predict(base_learners, meta_model, X_test_trees, X_test_lr)

# ==============================================================================
# OPTIMAL THRESHOLD SELECTION (Youden's J on validation set)
# ==============================================================================
print("\n--- Finding Optimal Classification Threshold ---")

fpr_val, tpr_val, thresholds_val = roc_curve(y_val, val_proba)
optimal_idx       = np.argmax(tpr_val - fpr_val)
optimal_threshold = thresholds_val[optimal_idx]
print(f"Optimal threshold (from validation set): {optimal_threshold:.4f}")

val_pred  = (val_proba  > optimal_threshold).astype(int)
test_pred = (test_proba > optimal_threshold).astype(int)


In [ ]:
# ==============================================================================
# STEP 7: EVALUATION
# ==============================================================================
print("\n" + "="*80)
print("STEP 7: EVALUATION")
print("="*80)

def evaluate_model(y_true, y_pred, y_proba, set_name="", threshold=0.5):
    """Standard classification evaluation: accuracy, precision, recall, F1, AUC-ROC."""
    print(f"\n--- {set_name} Evaluation (Threshold={threshold:.4f}) ---")
    print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  F1 Score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  AUC-ROC:   {roc_auc_score(y_true, y_proba):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Down/Flat', 'Up'], zero_division=0))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auc_roc':   roc_auc_score(y_true, y_proba)
    }

val_metrics  = evaluate_model(y_val,  val_pred,  val_proba,  "Validation Set",         optimal_threshold)
test_metrics = evaluate_model(y_test, test_pred, test_proba, "Test Set (Out-of-Sample)", optimal_threshold)

print("\n--- Base Learner Comparison (Test Set) ---")
for name, proba in test_base_preds.items():
    pred = (proba > optimal_threshold).astype(int)
    print(f"  {name}: AUC={roc_auc_score(y_test, proba):.4f}, Accuracy={accuracy_score(y_test, pred):.4f}")
print(f"\n  ENSEMBLE: AUC={test_metrics['auc_roc']:.4f}, Accuracy={test_metrics['accuracy']:.4f}")

# ==============================================================================
# SHAP INTERPRETABILITY
# ==============================================================================
print("\n" + "="*80)
print("SHAP ANALYSIS")
print("="*80)

# Tree-based models — TreeExplainer (unscaled data)
# Logistic Regression — LinearExplainer (StandardScaler data)
# [REDACTED — SHAP explainer calls and beeswarm/dependence plot code omitted]
# Standard pattern:
#   explainer = shap.TreeExplainer(model)
#   shap_values = explainer.shap_values(X_test)
#   shap.summary_plot(shap_values, X_test, feature_names=selected_features)
print("SHAP analysis: implement using shap.TreeExplainer / shap.LinearExplainer per model.")


### Key Results Summary

Classification and trading performance metrics are printed by the evaluation cell above.

SHAP analysis revealed that market microstructure, flow, and regime features
dominate directional prediction — with non-linear, regime-dependent relationships
rather than simple monotonic signals.


## Backtest

In [ ]:
# ==============================================================================
# STEP 7 (cont): BACKTESTING
# ==============================================================================
print("\n--- Backtesting Trading Strategy ---")

def backtest_strategy(dates, returns, proba, threshold):
    """
    Long-only strategy: go long when ensemble probability >= threshold, else cash.
    Returns a daily backtest DataFrame and a metrics dict.

    Metrics returned: total_return, annual_return, volatility, sharpe,
                      max_drawdown, win_rate, bh_total_return, bh_sharpe, bh_max_drawdown
    """
    # [REDACTED — daily P&L, cumulative curve, and metrics calculation omitted]
    raise NotImplementedError("Implement signal generation, cumulative returns, Sharpe, and max drawdown calculation")

test_returns = df_features.loc[dates_test, 'Next_Return'].values

df_backtest, backtest_metrics = backtest_strategy(
    dates=dates_test,
    returns=test_returns,
    proba=test_proba,
    threshold=optimal_threshold
)


In [ ]:
# =============================================================================
# PLOT ENTRY / EXIT POINTS ON PRICE + EQUITY CURVE (Plotly)
# Assumes you already have:
#   - df_backtest (from backtest_strategy) with columns: Signal, Strategy_Cumulative, BuyHold_Cumulative
#   - df_features (has 'Close' and 'Next_Return' etc.)
#   - dates_test (index for test period)
# =============================================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Align price series to backtest dates ---
price = df_features.loc[df_backtest.index, 'Close'].copy()

# --- Identify entries and exits ---
# Entry = Signal goes 0 -> 1
# Exit  = Signal goes 1 -> 0
signal = df_backtest['Signal'].astype(int)

entries = (signal.shift(1, fill_value=0) == 0) & (signal == 1)
exits   = (signal.shift(1, fill_value=0) == 1) & (signal == 0)

entry_dates = df_backtest.index[entries]
exit_dates  = df_backtest.index[exits]

# Optional: if you end test period still in a trade, mark the last day as an "exit"
if signal.iloc[-1] == 1:
    exit_dates = exit_dates.append(pd.Index([df_backtest.index[-1]]))

# --- Build a 2-row chart: Price w/ markers + Equity curve ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.55, 0.45],
    subplot_titles=("SPX Close with Strategy Entry/Exit", "Equity Curve (Strategy vs Buy & Hold)")
)

# Row 1: Price
fig.add_trace(
    go.Scatter(x=price.index, y=price.values, mode='lines', name='SPX Close'),
    row=1, col=1
)

# Entry markers
fig.add_trace(
    go.Scatter(
        x=entry_dates,
        y=price.loc[entry_dates],
        mode='markers',
        name='Entry (Long On)',
        marker=dict(symbol='triangle-up', size=10),
        hovertemplate="Entry<br>%{x|%Y-%m-%d}<br>Close: %{y:.2f}<extra></extra>"
    ),
    row=1, col=1
)

# Exit markers
fig.add_trace(
    go.Scatter(
        x=exit_dates,
        y=price.loc[exit_dates],
        mode='markers',
        name='Exit (Flat)',
        marker=dict(symbol='triangle-down', size=10),
        hovertemplate="Exit<br>%{x|%Y-%m-%d}<br>Close: %{y:.2f}<extra></extra>"
    ),
    row=1, col=1
)

# Optional: shade "in-position" regions (can be heavy if many trades; keep if you want)
# This creates vertical rectangles for each trade period.
in_trade = signal == 1
trade_starts = df_backtest.index[(~in_trade.shift(1, fill_value=False)) & in_trade]
trade_ends   = df_backtest.index[(in_trade.shift(1, fill_value=False)) & (~in_trade)]
if len(trade_starts) > len(trade_ends):
    trade_ends = trade_ends.append(pd.Index([df_backtest.index[-1]]))

for s, e in zip(trade_starts, trade_ends):
    fig.add_vrect(x0=s, x1=e, row=1, col=1, opacity=0.12, line_width=0)

# Row 2: Equity curves
fig.add_trace(
    go.Scatter(
        x=df_backtest.index,
        y=df_backtest['Strategy_Cumulative'],
        mode='lines',
        name='Strategy Equity'
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_backtest.index,
        y=df_backtest['BuyHold_Cumulative'],
        mode='lines',
        name='Buy & Hold Equity'
    ),
    row=2, col=1
)

fig.update_layout(
    height=850,
    width=1100,
    title="Strategy Entries/Exits + Equity Curve (Test Set)",
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.update_yaxes(title_text="SPX Close", row=1, col=1)
fig.update_yaxes(title_text="Cumulative Return", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()


## Strategy 2

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------
# 0) Align test data
# ----------------------------
df_test = pd.DataFrame(index=dates_test)
df_test["Next_Return"] = df_features.loc[dates_test, "Next_Return"].values
df_test["Proba"]       = test_proba
df_test["y_true"]      = y_test

n_before = len(df_test)
df_test = df_test.dropna(subset=["Next_Return", "Proba"])
print(f"Dropped {n_before - len(df_test)} rows due to NaNs. Remaining: {len(df_test)}")

dates_clean   = df_test.index
returns_clean = df_test["Next_Return"].to_numpy()
proba_clean   = df_test["Proba"].to_numpy()
returns_s     = pd.Series(returns_clean, index=dates_clean)

# ----------------------------
# 1) Base signal
# ----------------------------
signal = (proba_clean >= optimal_threshold).astype(int)
print("Signal rate (days in market):", signal.mean())

# ----------------------------
# 2) Volatility-targeted position sizing
# Uses lagged realized vol to scale exposure toward a target annualized volatility.
# No look-ahead: vol estimated on t, applied on t+1.
# [REDACTED — vol window, target vol, and max leverage parameters omitted]
# [REDACTED — realized vol calculation and position scaler omitted]
# ----------------------------

# position_pure = signal * vol_scaler  # [REDACTED]
position_pure = signal.astype(float)   # placeholder — replace with vol-targeted sizing

# ----------------------------
# 3) Backtest
# ----------------------------
df_bt = pd.DataFrame(index=dates_clean)
df_bt["Actual_Return"]       = returns_s.values
df_bt["Signal"]              = signal
df_bt["Position"]            = position_pure
df_bt["Strategy_Return"]     = df_bt["Position"] * df_bt["Actual_Return"]
df_bt["BuyHold_Return"]      = df_bt["Actual_Return"]
df_bt["Strategy_Cumulative"] = (1 + df_bt["Strategy_Return"]).cumprod()
df_bt["BuyHold_Cumulative"]  = (1 + df_bt["BuyHold_Return"]).cumprod()

# [REDACTED — Sharpe, max drawdown, win rate metrics omitted]
print("Strategy 2 backtest complete. Compute metrics from df_bt.")


## Final conclusions

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# Build one aligned backtest frame (same dates for all 3)
# =========================================================
df_cmp = pd.DataFrame(index=dates_clean)
df_cmp["Return"] = returns_clean
df_cmp["Proba"]  = proba_clean

# ----------------------------
# Strategy 1: ML only (binary in/out after optimal threshold)
# ----------------------------
df_cmp["Pos_S1_ML"] = (df_cmp["Proba"] >= optimal_threshold).astype(float)

# ----------------------------
# Strategy 2: ML + Lever 3 (PURE vol targeting)
# ----------------------------
df_cmp["Pos_S2_VolTarget"] = np.asarray(position_pure, dtype=float)

# ----------------------------
# Buy & Hold (always 1x exposure)
# ----------------------------
df_cmp["Pos_BuyHold"] = 1.0

# =========================================================
# Compute returns and cumulative curves
# =========================================================
df_cmp["Ret_BuyHold"] = df_cmp["Pos_BuyHold"] * df_cmp["Return"]
df_cmp["Ret_S1"]      = df_cmp["Pos_S1_ML"]  * df_cmp["Return"]
df_cmp["Ret_S2"]      = df_cmp["Pos_S2_VolTarget"] * df_cmp["Return"]

df_cmp["Cum_BuyHold"] = (1 + df_cmp["Ret_BuyHold"]).cumprod()
df_cmp["Cum_S1"]      = (1 + df_cmp["Ret_S1"]).cumprod()
df_cmp["Cum_S2"]      = (1 + df_cmp["Ret_S2"]).cumprod()

# =========================================================
# Drawdowns (in %)
# =========================================================
def drawdown(series: pd.Series) -> pd.Series:
    peak = series.cummax()
    return (series / peak - 1.0) * 100.0

df_cmp["DD_BuyHold"] = drawdown(df_cmp["Cum_BuyHold"])
df_cmp["DD_S1"]      = drawdown(df_cmp["Cum_S1"])
df_cmp["DD_S2"]      = drawdown(df_cmp["Cum_S2"])

# =========================================================
# Plot 1: Cumulative return (all 3 together)
# =========================================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_cmp.index, y=df_cmp["Cum_BuyHold"], mode="lines", name="Buy & Hold"))
fig.add_trace(go.Scatter(x=df_cmp.index, y=df_cmp["Cum_S1"],      mode="lines", name="Strategy 1 (ML Threshold)"))
fig.add_trace(go.Scatter(x=df_cmp.index, y=df_cmp["Cum_S2"],      mode="lines", name="Strategy 2 (ML + Vol Target)"))

fig.update_layout(
    title="Cumulative Return Comparison",
    xaxis_title="Date",
    yaxis_title="Cumulative Growth (1.0 = start)",
    height=520,
    width=950
)
fig.show()

# =========================================================
# Plot 2: Drawdowns (all 3 together)
# =========================================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_cmp.index, y=df_cmp["DD_BuyHold"], mode="lines", name="Buy & Hold"))
fig.add_trace(go.Scatter(x=df_cmp.index, y=df_cmp["DD_S1"],      mode="lines", name="Strategy 1 (ML)"))
fig.add_trace(go.Scatter(x=df_cmp.index, y=df_cmp["DD_S2"],      mode="lines", name="Strategy 2 (ML + Vol Target)"))

fig.update_layout(
    title="Drawdown Comparison (%)",
    xaxis_title="Date",
    yaxis_title="Drawdown (%)",
    height=520,
    width=950
)
fig.show()

print("\nMax Drawdowns:")
print("Buy & Hold:", df_cmp["DD_BuyHold"].min())
print("Strategy 1:", df_cmp["DD_S1"].min())
print("Strategy 2:", df_cmp["DD_S2"].min())


### With TC

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# TRANSACTION COST MODEL
# =========================================================
INSTRUMENT        = "SPY"   # "SPY" (bps per trade) or "ES" (ticks + fees)
SPY_BPS_PER_TRADE = None    # [REDACTED] — bps charged per unit of turnover
ES_SLIPPAGE_TICKS = None    # [REDACTED]
ES_FEES_TICKS     = None    # [REDACTED]
ES_TICK_SIZE      = 0.25    # ES tick size in index points

# =========================================================
# Build aligned frame
# =========================================================
df_cmp = pd.DataFrame(index=dates_clean)
df_cmp["Return"] = returns_clean
df_cmp["Proba"]  = proba_clean

df_cmp["Pos_S1"] = (df_cmp["Proba"] >= optimal_threshold).astype(float)
df_cmp["Pos_S2"] = np.asarray(position_pure, dtype=float)
df_cmp["Pos_BH"] = 1.0

def add_tc_and_compute_net(df, pos_col, ret_col="Return", instrument="SPY"):
    """
    Apply transaction costs on turnover (|Δposition|) and compute net P&L.
    Supports SPY (bps model) and ES (ticks + fees model).
    Returns df with: Turnover, TC, Gross, Net, Cum_Gross, Cum_Net, DD_Net columns.
    """
    # [REDACTED — TC calculation, cumulative curve, and drawdown logic omitted]
    raise NotImplementedError("Implement turnover-based TC engine for SPY (bps) or ES (ticks)")

def summarize(df_net, name):
    """
    Compute summary statistics from a net-return DataFrame.
    Returns: total return, annualized return, vol, Sharpe, max drawdown,
             turnover, trade count, TC paid, and calendar-year metrics.
    """
    # [REDACTED]
    raise NotImplementedError("Implement performance summary statistics")

cols_close = ["Close"] if "Close" in df_cmp.columns else []

df_bh = add_tc_and_compute_net(
    df_cmp[["Return", "Pos_BH"] + cols_close].rename(columns={"Pos_BH": "Pos"}),
    pos_col="Pos", instrument=INSTRUMENT
)
df_s1 = add_tc_and_compute_net(
    df_cmp[["Return", "Pos_S1"] + cols_close].rename(columns={"Pos_S1": "Pos"}),
    pos_col="Pos", instrument=INSTRUMENT
)
df_s2 = add_tc_and_compute_net(
    df_cmp[["Return", "Pos_S2"] + cols_close].rename(columns={"Pos_S2": "Pos"}),
    pos_col="Pos", instrument=INSTRUMENT
)

# Summary table
df_tc_compare = pd.DataFrame([
    summarize(df_bh, "Buy & Hold (net TC)"),
    summarize(df_s1, "Strategy 1: Ensemble ML (net TC)"),
    summarize(df_s2, "Strategy 2: ML + Vol Target (net TC)"),
]).sort_values("AnnReturn_%", ascending=False)

print(f"\nTransaction Cost Model: {INSTRUMENT}")
display(df_tc_compare)


## Logistic Regression only

In [ ]:
# =========================================================
# LOGISTIC-ONLY SIGNAL (instead of ensemble/meta)
# Compare vs Buy&Hold, Strategy 1 (ensemble ML), Strategy 2 (vol target)
# =========================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score

# ----------------------------
# 0) Guardrails / assumptions
# ----------------------------
required = ["df_features", "CONFIG", "dates_clean", "returns_clean", "proba_clean", "optimal_threshold", "position_pure"]
missing = [v for v in required if v not in globals()]
if missing:
    raise ValueError(f"Missing required variables in memory: {missing}\n"
                     f"Run your main notebook cells first (features/splits + ensemble backtest + Strategy2 position_pure).")

# ----------------------------
# 0.5) IMPORTANT: Match the HTML logistic feature set (RFECV-selected 10)
# ----------------------------
# RFECV-selected features from Stage 3 of the funnel.
# [REDACTED — populate from your own RFECV run via `selected_features`]
HTML_LOGIT_FEATURES = []  # [REDACTED]

# If your notebook already has `selected_features` from the HTML run, use it (as long as it's length 10).
# Otherwise fallback to the HTML list above.
if "selected_features" in globals() and isinstance(selected_features, (list, tuple)) and len(selected_features) == 10:
    logit_features = list(selected_features)
else:
    logit_features = HTML_LOGIT_FEATURES

# sanity check: make sure all are present
missing_feats = [c for c in logit_features if c not in df_features.columns]
if missing_feats:
    raise ValueError(f"These HTML logistic features are missing from df_features: {missing_feats}")

# ----------------------------
# 1) Recreate the same train/val/test split used in the HTML
# ----------------------------
X_all = df_features[logit_features].values          # <-- CHANGED (was feature_cols)
y_all = df_features["Target"].values
dates_all = df_features.index

n_samples = len(X_all)
test_start_idx = int(n_samples * (1 - CONFIG["test_size"]))
val_start_idx  = int(n_samples * (1 - CONFIG["test_size"] - CONFIG["val_size"]))

X_train_raw = X_all[:val_start_idx]
y_train     = y_all[:val_start_idx]
dates_train = dates_all[:val_start_idx]

X_val_raw = X_all[val_start_idx:test_start_idx]
y_val     = y_all[val_start_idx:test_start_idx]
dates_val = dates_all[val_start_idx:test_start_idx]

X_test_raw = X_all[test_start_idx:]
y_test     = y_all[test_start_idx:]
dates_test = dates_all[test_start_idx:]

# ----------------------------
# 2) StandardScaler + Logistic GridSearch (elasticnet + saga)
# ----------------------------
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_raw)
X_val_std   = scaler.transform(X_val_raw)
X_test_std  = scaler.transform(X_test_raw)

tscv = TimeSeriesSplit(n_splits=5)

lr = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    class_weight="balanced",
    max_iter=1000,
    random_state=CONFIG["random_state"],
)

lr_params = {
    "C": [0.01, 0.1, 1, 10],
    "l1_ratio": [0.1, 0.5, 0.9],
}

lr_grid = GridSearchCV(lr, lr_params, cv=tscv, scoring="roc_auc", n_jobs=-1)
lr_grid.fit(X_train_std, y_train)

logit_best = lr_grid.best_estimator_

# Probabilities on VAL/TEST
val_proba_logit  = logit_best.predict_proba(X_val_std)[:, 1]
test_proba_logit = logit_best.predict_proba(X_test_std)[:, 1]

# ---- CHANGED/ADDED: print the TRUE test AUC on the full test set (same concept as HTML)
test_auc = roc_auc_score(y_test, test_proba_logit)

print(f"[Logistic] Features used (n={len(logit_features)}): {logit_features}")
print(f"[Logistic] Best params: {lr_grid.best_params_}")
print(f"[Logistic] CV AUC (on TRAIN via TSCV): {lr_grid.best_score_:.4f}")
print(f"[Logistic] TEST AUC (full test set): {test_auc:.4f}")
print(f"[Logistic] Test proba stats: min={test_proba_logit.min():.4f} max={test_proba_logit.max():.4f} mean={test_proba_logit.mean():.4f}")

# ----------------------------
# 3) Logistic-specific optimal threshold (Youden J on validation)
# ----------------------------
fpr_val, tpr_val, thresholds_val = roc_curve(y_val, val_proba_logit)
optimal_idx_logit = np.argmax(tpr_val - fpr_val)
optimal_threshold_logit = thresholds_val[optimal_idx_logit]
print(f"[Logistic] Optimal threshold (from validation): {optimal_threshold_logit:.4f}")

# ----------------------------
# 4) Align logistic probs to your existing backtest alignment (dates_clean / returns_clean)
# ----------------------------
df_logit = pd.DataFrame(index=dates_test)
df_logit["Proba_Logit"] = test_proba_logit

df_logit = df_logit.reindex(dates_clean)
if df_logit["Proba_Logit"].isna().any():
    keep = ~df_logit["Proba_Logit"].isna()
    print(f"[WARN] Dropping {int((~keep).sum())} rows from dates_clean due to logistic proba NaNs.")
    df_logit = df_logit.loc[keep]

final_idx = df_logit.index
returns_final    = pd.Series(returns_clean, index=dates_clean).reindex(final_idx).to_numpy()
proba_ens_final  = pd.Series(proba_clean, index=dates_clean).reindex(final_idx).to_numpy()
pos_s2_final     = pd.Series(np.asarray(position_pure, dtype=float), index=dates_clean).reindex(final_idx).to_numpy()

# ----------------------------
# 5) Build comparison frame
# ----------------------------
df_cmp = pd.DataFrame(index=final_idx)
df_cmp["Return"] = returns_final
df_cmp["Proba_Ensemble"] = proba_ens_final
df_cmp["Proba_Logit"] = df_logit["Proba_Logit"].to_numpy()

df_cmp["Pos_BH"]    = 1.0
df_cmp["Pos_S1"]    = (df_cmp["Proba_Ensemble"] >= optimal_threshold).astype(float)
df_cmp["Pos_S2"]    = np.asarray(pos_s2_final, dtype=float)
df_cmp["Pos_Logit"] = (df_cmp["Proba_Logit"] >= optimal_threshold_logit).astype(float)

# =========================================================
# 6) Transaction costs + net returns (same engine as your HTML)
# =========================================================
INSTRUMENT        = "SPY"   # "SPY" bps OR "ES" ticks+fees
SPY_BPS_PER_TRADE = None    # [REDACTED]
ES_SLIPPAGE_TICKS = None    # [REDACTED]
ES_FEES_TICKS     = None    # [REDACTED]
ES_TICK_SIZE      = 0.25

def add_tc_and_compute_net(df, pos_col, ret_col="Return", instrument="SPY"):
    """
    Apply transaction costs on turnover and compute net P&L.
    [REDACTED — implementation omitted, see TC engine cell]
    """
    raise NotImplementedError("Import or reuse the add_tc_and_compute_net defined in the TC engine cell above")

# [REDACTED — calendar_year_return, calendar_year_max_dd, summarize helpers omitted]
# These utility functions compute calendar-year returns, drawdowns, and full
# performance summaries. Implement or import from your backtest utilities module.

df_bh    = add_tc_and_compute_net(df_cmp[["Return","Pos_BH"]].rename(columns={"Pos_BH":"Pos"}), "Pos", instrument=INSTRUMENT)
df_s1    = add_tc_and_compute_net(df_cmp[["Return","Pos_S1"]].rename(columns={"Pos_S1":"Pos"}), "Pos", instrument=INSTRUMENT)
df_s2    = add_tc_and_compute_net(df_cmp[["Return","Pos_S2"]].rename(columns={"Pos_S2":"Pos"}), "Pos", instrument=INSTRUMENT)
df_logit = add_tc_and_compute_net(df_cmp[["Return","Pos_Logit"]].rename(columns={"Pos_Logit":"Pos"}), "Pos", instrument=INSTRUMENT)

df_tc_compare = pd.DataFrame([
    summarize(df_bh,    "Buy & Hold (net TC)"),
    summarize(df_s1,    "Strategy 1: Ensemble ML (net TC)"),
    summarize(df_s2,    "Strategy 2: ML + Vol Target (net TC)"),
    summarize(df_logit, "Logistic-only: threshold ML (net TC)"),
]).sort_values("AnnReturn_%", ascending=False)

print(f"\nTransaction Cost Model: {INSTRUMENT}")
display(df_tc_compare)

# =========================================================
# 7) Plots: cumulative net return + net drawdown
# =========================================================
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=df_bh.index,    y=df_bh["Cum_Net"],    mode="lines", name="Buy & Hold"))
fig1.add_trace(go.Scatter(x=df_s1.index,    y=df_s1["Cum_Net"],    mode="lines", name="Strategy 1 (Ensemble)"))
fig1.add_trace(go.Scatter(x=df_s2.index,    y=df_s2["Cum_Net"],    mode="lines", name="Strategy 2 (Vol Target)"))
fig1.add_trace(go.Scatter(x=df_logit.index, y=df_logit["Cum_Net"], mode="lines", name="Logistic-only"))
fig1.update_layout(
    title="Cumulative Net Equity (after TC)",
    xaxis_title="Date",
    yaxis_title="Equity (starting at 1.0)",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_bh.index,    y=df_bh["DD_Net"],    mode="lines", name="Buy & Hold"))
fig2.add_trace(go.Scatter(x=df_s1.index,    y=df_s1["DD_Net"],    mode="lines", name="Strategy 1 (Ensemble)"))
fig2.add_trace(go.Scatter(x=df_s2.index,    y=df_s2["DD_Net"],    mode="lines", name="Strategy 2 (Vol Target)"))
fig2.add_trace(go.Scatter(x=df_logit.index, y=df_logit["DD_Net"], mode="lines", name="Logistic-only"))
fig2.update_layout(
    title="Net Drawdown (%) (after TC)",
    xaxis_title="Date",
    yaxis_title="Drawdown (%)",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig2.show()


### Logistic - Vol targetting

In [ ]:
# =========================================================
# Logistic - Vol Targetting (Strategy 3)
# (same vol targeting engine as your Strategy 2; safe parameter fallback)
# OUTPUT: df_logit_vol_net
# =========================================================

import numpy as np
import pandas as pd

# --- Resolve vol params (prefer existing globals used by Strategy 2) ---
VOL_WINDOW = (
    globals().get("VOL_WINDOW")
    or globals().get("vol_window")
    or (CONFIG.get("vol_window") if "CONFIG" in globals() else None)
    or None  # [REDACTED]
)
TARGET_VOL = (
    globals().get("TARGET_VOL")
    or globals().get("target_vol")
    or (CONFIG.get("target_vol") if "CONFIG" in globals() else None)
    or None  # [REDACTED]
)
LEVERAGE_CAP = (
    globals().get("LEVERAGE_CAP")
    or globals().get("leverage_cap")
    or (CONFIG.get("leverage_cap") if "CONFIG" in globals() else None)
    or None  # [REDACTED]
)

ANN_FACTOR = 252

# --- Build frame ---
df_s3 = pd.DataFrame(index=final_idx)
df_s3["Return"] = returns_final
df_s3["Proba_Logit"] = df_cmp.loc[final_idx, "Proba_Logit"]

# --- Continuous centered signal ---
df_s3["Signal"] = df_s3["Proba_Logit"] - 0.5

# --- Realized vol + leverage ---
df_s3["RealizedVol"] = df_s3["Return"].rolling(VOL_WINDOW).std() * np.sqrt(ANN_FACTOR)
df_s3["Leverage"] = (TARGET_VOL / df_s3["RealizedVol"]).clip(0.0, LEVERAGE_CAP).fillna(0.0)

# --- Final position + net backtest (with TC) ---
df_s3["Pos"] = (df_s3["Signal"] * df_s3["Leverage"]).clip(-LEVERAGE_CAP, LEVERAGE_CAP)

df_logit_vol_net = add_tc_and_compute_net(
    df_s3[["Return", "Pos"]],
    pos_col="Pos",
    instrument=INSTRUMENT
)

### Logistic with Confidence based position sizing

In [ ]:
# =========================================================
# Logistic - Position Sizing (Strategy 4)
# OUTPUT: df_logit_possize_net
# =========================================================

import numpy as np
import pandas as pd

MAX_POS = 1.0        # max exposure
POWER   = None       # [REDACTED] — >1 dampens weak signals, <1 amplifies

df_s4 = pd.DataFrame(index=final_idx)
df_s4["Return"] = returns_final
df_s4["Proba_Logit"] = df_cmp.loc[final_idx, "Proba_Logit"]

signal = df_s4["Proba_Logit"] - 0.5
scaled = np.sign(signal) * (np.abs(signal) ** POWER)

df_s4["Pos"] = (MAX_POS * scaled).clip(-MAX_POS, MAX_POS)

df_logit_possize_net = add_tc_and_compute_net(
    df_s4[["Return", "Pos"]],
    pos_col="Pos",
    instrument=INSTRUMENT
)

### Logistic with Confidence filter 

In [ ]:
# =========================================================
# Logistic - Confidence Filter (Strategy 5)
# OUTPUT: df_logit_conffilter_net
# =========================================================

import numpy as np
import pandas as pd

CONF_THRESH = None  # [REDACTED] — trade only if |p-0.5| >= threshold

df_s5 = pd.DataFrame(index=final_idx)
df_s5["Return"] = returns_final
df_s5["Proba_Logit"] = df_cmp.loc[final_idx, "Proba_Logit"]

signal = df_s5["Proba_Logit"] - 0.5

df_s5["Pos"] = 0.0
df_s5.loc[signal >=  CONF_THRESH, "Pos"] =  1.0
df_s5.loc[signal <= -CONF_THRESH, "Pos"] = -1.0

df_logit_conffilter_net = add_tc_and_compute_net(
    df_s5[["Return", "Pos"]],
    pos_col="Pos",
    instrument=INSTRUMENT
)

## Final Conclusions

In [ ]:
# =========================================================
# ONE CELL — 2 PLOTS (Cum Net + Net Drawdown)
# Buy&Hold, Logistic-only, Logistic-VolTarget, Logistic-PosSizing, Logistic-ConfFilter
# Requires: df_bh, df_logit (logistic-only threshold), df_logit_vol_net, df_logit_possize_net, df_logit_conffilter_net
# =========================================================

import plotly.graph_objects as go

# If your logistic-only net df is named differently, set it here:
# e.g. df_logit_only_net = df_logit
df_logit_only_net = df_logit

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=df_bh.index, y=df_bh["Cum_Net"], mode="lines", name="Buy & Hold"))
fig1.add_trace(go.Scatter(x=df_logit_only_net.index, y=df_logit_only_net["Cum_Net"], mode="lines", name="Logistic-only"))
fig1.add_trace(go.Scatter(x=df_logit_vol_net.index, y=df_logit_vol_net["Cum_Net"], mode="lines", name="Logistic - Vol targetting"))
fig1.add_trace(go.Scatter(x=df_logit_possize_net.index, y=df_logit_possize_net["Cum_Net"], mode="lines", name="Logistic - Position Sizing"))
fig1.add_trace(go.Scatter(x=df_logit_conffilter_net.index, y=df_logit_conffilter_net["Cum_Net"], mode="lines", name="Logistic - Confidence Filter"))
fig1.update_layout(
    title="Cumulative Net Equity (after TC)",
    xaxis_title="Date",
    yaxis_title="Equity (start=1.0)",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_bh.index, y=df_bh["DD_Net"], mode="lines", name="Buy & Hold"))
fig2.add_trace(go.Scatter(x=df_logit_only_net.index, y=df_logit_only_net["DD_Net"], mode="lines", name="Logistic-only"))
fig2.add_trace(go.Scatter(x=df_logit_vol_net.index, y=df_logit_vol_net["DD_Net"], mode="lines", name="Logistic - Vol targetting"))
fig2.add_trace(go.Scatter(x=df_logit_possize_net.index, y=df_logit_possize_net["DD_Net"], mode="lines", name="Logistic - Position Sizing"))
fig2.add_trace(go.Scatter(x=df_logit_conffilter_net.index, y=df_logit_conffilter_net["DD_Net"], mode="lines", name="Logistic - Confidence Filter"))
fig2.update_layout(
    title="Net Drawdown (%) (after TC)",
    xaxis_title="Date",
    yaxis_title="Drawdown (%)",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig2.show()


In [ ]:
# =========================================================
# ONE CELL — Summary stats table
# Buy & Hold, Strategy 1, Strategy 2, and ALL Logistic variants:
# Logistic-only, Logistic-VolTarget, Logistic-PosSizing, Logistic-ConfFilter
# =========================================================

import pandas as pd

# Map names to your already-computed net DFs:
# df_s1 and df_s2 from earlier, plus df_logit variants from above.
# If your Strategy 1/2 DFs have different variable names, update here.

rows = [
    summarize(df_bh, "Buy & Hold (net TC)"),
    summarize(df_s1, "Strategy 1: Ensemble ML (net TC)"),
    summarize(df_s2, "Strategy 2: ML + Vol Target (net TC)"),
]

# Logistic-only (threshold) — set alias safely
df_logit_only_net = df_logit

rows += [
    summarize(df_logit_only_net, "Logistic-only (net TC)"),
    summarize(df_logit_vol_net, "Logistic - Vol targetting (net TC)"),
    summarize(df_logit_possize_net, "Logistic - Position Sizing (net TC)"),
    summarize(df_logit_conffilter_net, "Logistic - Confidence Filter (net TC)"),
]

df_summary = pd.DataFrame(rows).sort_values("AnnReturn_%", ascending=False)
display(df_summary)


In [ ]:
# =========================================================
# ONE CELL — Summary stats table
# Buy & Hold, Strategy 1, Strategy 2, Logistic-only
# Includes:
# - Directional Win Rate (active days)
# - 2025 Calendar-Year Return
# - 2025 Calendar-Year Max Drawdown
# =========================================================

import pandas as pd
import numpy as np

def directional_win_rate(df_net):
    """
    Directional win rate on active days:
    win if Pos * Return > 0.
    Matches HTML / ML-style directional accuracy.
    """
    pos = df_net["Pos"].astype(float).values
    ret = df_net["Return"].astype(float).values

    active = pos != 0
    if active.sum() == 0:
        return np.nan

    return (pos[active] * ret[active] > 0).mean()

rows = []

# Buy & Hold (no win rate)
bh = summarize(df_bh, "Buy & Hold (net TC)")
bh["WinRate_%"] = np.nan
rows.append(bh)

# Strategy 1 — Ensemble ML
s1 = summarize(df_s1, "Strategy 1: Ensemble ML (net TC)")
s1["WinRate_%"] = directional_win_rate(df_s1) * 100
rows.append(s1)

# Strategy 2 — ML + Vol Target
s2 = summarize(df_s2, "Strategy 2: ML + Vol Target (net TC)")
s2["WinRate_%"] = directional_win_rate(df_s2) * 100
rows.append(s2)

# Logistic-only
lg = summarize(df_logit, "Logistic-only (net TC)")
lg["WinRate_%"] = directional_win_rate(df_logit) * 100
rows.append(lg)

df_summary_wr = (
    pd.DataFrame(rows)
    .loc[:, [
        "Strategy",
        "TotalReturn_%",
        "Sharpe",
        "MaxDD_%",
        "WinRate_%",
        "CY2025_Return_%",
        "CY2025_MaxDD_%",
        "Trades",
        "Turnover",
        "TC_Paid_%"
    ]]
    .sort_values("TotalReturn_%", ascending=False)
)

display(df_summary_wr)
